# 多类型异构车队车辆路径问题 (MTHFVRP) 数据生成

本 Notebook 旨在演示如何使用 `MTHFVRPGenerator` 生成不同变体的车辆路径问题 (VRP) 数据，并将生成的数据保存为 JSONL 格式。

主要内容包括：
1. **支持的问题变体介绍**：详细说明生成器支持的 VRP 变体及其配置。
2. **工具函数定义**：定义用于格式转换和保存文件的辅助函数。
3. **生成与保存流程**：演示如何生成数据、计算问题签名并保存到文件。

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import json
from typing import Dict, Any

# 将项目根目录添加到 Python 路径，以便导入项目模块
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    print(f"Adding project root to Python path: {project_root}")
    sys.path.insert(0, project_root)

# 导入生成器
from implement.generator import MTHFVRPGenerator

Adding project root to Python path: /home/huangsh/SF_NCO/mvhfvrp_20260204


## 1. 支持的问题变体 (Variants)

`MTHFVRPGenerator` 通过组合不同的约束特征来支持多种 VRP 变体。主要特征包括：

- **O (Open)**: 开放式路径，车辆不需要返回车场。
- **TW (Time Window)**: 时间窗约束。
- **L (Limit)**: 距离限制约束。
- **B (Backhaul)**: 回程约束（混合了送货和取货需求）。
- **HF (Heterogeneous Fleet)**: 异构车队（不同容量和成本的车辆）。

您可以通过 `variant_preset` 参数直接指定预定义的变体名称。以下是支持的主要变体类型：

### 基础变体 (同构车队)
这些变体默认不包含异构车队特征 (HF=0)。

| 变体名称 | 描述 | 特征组合 |
| :--- | :--- | :--- |
| `cvrp` | 经典容量限制车辆路径问题 | 基础 (无额外约束) |
| `ovrp` | 开放式 VRP | Open |
| `vrptw` | 带时间窗 VRP | Time Window |
| `vrpb` | 带回程需求的 VRP | Backhaul |
| `vrpl` | 带距离限制的 VRP | Distance Limit | 
| ... | (支持更多组合，如 `vrpbl`, `ovrpl`, `ovrptw` 等) | ... |

### 异构车队变体 (HF)
这些变体在基础变体之上增加了异构车队特征 (HF=1)。变体名称通常以 `hf` 开头。

| 变体名称 | 描述 | 特征组合 |
| :--- | :--- | :--- |
| `hfcvrp` | 异构车队 CVRP | HF |
| `hfvrptw` | 异构车队带时间窗 VRP | HF + Time Window |
| `hfovrp` | 异构车队开放式 VRP | HF + Open |
| `hfvrpbl` | 异构车队带回程和限制 VRP | HF + Backhaul + Limit |
| ... | (对应上述基础变体的 HF 版本) | ... |

您可以直接在代码中查看所有支持的预设：

In [2]:
from implement.generator import VARIANT_GENERATION_PRESETS

print("所有支持的变体预设 (Keys):")
for key, value in VARIANT_GENERATION_PRESETS.items():
    print(f"{key}: {value}")

所有支持的变体预设 (Keys):
no_hf_all: {'O': 0.5, 'TW': 0.5, 'L': 0.5, 'B': 0.5, 'HF': 0.0}
hf_all: {'O': 0.5, 'TW': 0.5, 'L': 0.5, 'B': 0.5, 'HF': 1.0}
cvrp: {'O': 0.0, 'TW': 0.0, 'L': 0.0, 'B': 0.0, 'HF': 0.0}
ovrp: {'O': 1.0, 'TW': 0.0, 'L': 0.0, 'B': 0.0, 'HF': 0.0}
vrpb: {'O': 0.0, 'TW': 0.0, 'L': 0.0, 'B': 1.0, 'HF': 0.0}
vrpl: {'O': 0.0, 'TW': 0.0, 'L': 1.0, 'B': 0.0, 'HF': 0.0}
vrptw: {'O': 0.0, 'TW': 1.0, 'L': 0.0, 'B': 0.0, 'HF': 0.0}
ovrptw: {'O': 1.0, 'TW': 1.0, 'L': 0.0, 'B': 0.0, 'HF': 0.0}
ovrpb: {'O': 1.0, 'TW': 0.0, 'L': 0.0, 'B': 1.0, 'HF': 0.0}
ovrpl: {'O': 1.0, 'TW': 0.0, 'L': 1.0, 'B': 0.0, 'HF': 0.0}
vrpbl: {'O': 0.0, 'TW': 0.0, 'L': 1.0, 'B': 1.0, 'HF': 0.0}
vrpbtw: {'O': 0.0, 'TW': 1.0, 'L': 0.0, 'B': 1.0, 'HF': 0.0}
vrpltw: {'O': 0.0, 'TW': 1.0, 'L': 1.0, 'B': 0.0, 'HF': 0.0}
ovrpbl: {'O': 1.0, 'TW': 0.0, 'L': 1.0, 'B': 1.0, 'HF': 0.0}
ovrpbtw: {'O': 1.0, 'TW': 1.0, 'L': 0.0, 'B': 1.0, 'HF': 0.0}
ovrpltw: {'O': 1.0, 'TW': 1.0, 'L': 1.0, 'B': 0.0, 'HF': 0.0}
vrpbltw: {'O'

## 2. 定义工具函数

为了将 PyTorch Tensor 格式的生成数据保存为通用的 JSONL 格式，我们需要定义数据转换和保存函数。

In [3]:
def convert_problem_to_dict(problem: Dict[str, torch.Tensor], batch_idx: int) -> Dict[str, Any]:
    """
    将 TensorDict 中的单个问题实例转换为 Python 字典格式，
    移除 Tensor 包装并转为 list 或 scalar。
    """
    single_problem = {}
    for key, value in problem.items():
        if isinstance(value, torch.Tensor):
            # 处理不同维度的 Tensor
            if value.dim() == 0:
                single_problem[key] = value.item()
            elif value.dim() == 1: 
                single_problem[key] = value[batch_idx].item()
            else: # dim >= 2
                single_problem[key] = value[batch_idx].cpu().numpy().tolist()
        else:
            single_problem[key] = value
            
    # 构建符合数据规范的目标字典
    # 注意：这里会根据 generator 实际输出的字段进行提取
    converted_dict = {
        "locs": single_problem.get("locs"),
        "demand_backhaul": single_problem.get("demand_backhaul"), 
        "demand_linehaul": single_problem.get("demand_linehaul"),
        "backhaul_class": single_problem.get("backhaul_class"),
        "distance_limit": single_problem.get("distance_limit"),
        "time_windows": single_problem.get("time_windows"),
        "service_time": single_problem.get("service_time"),
        "available_vehicles": single_problem.get("available_vehicles"),
        "vehicle_capacity": single_problem.get("vehicle_capacity"),
        "vehicle_fixed_cost": single_problem.get("vehicle_fixed_cost"),
        "variable_cost": single_problem.get("variable_cost"), # 注意名称可能略有不同，视generator输出而定
        "open_route": single_problem.get("open_route"),
        "speed": single_problem.get("speed"),
        "heterogeneous_fleet": single_problem.get("heterogeneous_fleet")
    }
    
    return converted_dict

def save_problems_to_jsonl(problem_batch, filename: str):
    """
    将批量生成的问题保存到 JSONL 文件。
    """
    # 确保目录存在
    directory = os.path.dirname(filename)
    if directory and not os.path.exists(directory):
        os.makedirs(directory, exist_ok=True)
        print(f"创建目录: {directory}")
    
    # 获取批量大小
    batch_size = problem_batch.batch_size[0] if hasattr(problem_batch, 'batch_size') else len(problem_batch['locs'])
    
    with open(filename, 'w', encoding='utf-8') as f:
        for i in range(batch_size):
            # 转换单个问题
            problem_dict = convert_problem_to_dict(problem_batch, i)
            # 写入文件
            f.write(json.dumps(problem_dict, ensure_ascii=False) + '\n')
    
    print(f"成功保存 {batch_size} 个实例到: {filename}")

## 3. 问题生成与保存流程

下面的代码演示了完整的流程：
1. **设置参数**：客户数量、车辆数量、问题数量。
2. **初始化生成器**：指定 `variant_preset`。
3. **生成数据**：调用生成器得到 batch 数据。
4. **生成签名**：打印第一个算例的第一个坐标和最后一个算例的第一个坐标，作为校验签名。
5. **保存文件**。

这里我们演示生成 `cvrp`, `vrptw` 和 `hfcvrp` 三种变体的示例。

In [4]:
import tqdm

# 生成配置
variants_to_generate = ["cvrp", "vrptw", "hfcvrp"]
nums_customers = 50   # 客户节点数 (不含 Depot)
num_veh = 10          # 车辆数
nums_problems = 1000   # 每个变体生成的实例数
output_dir = "../data" # 输出目录

for variant in tqdm.tqdm(variants_to_generate, desc="Generating Variants"):
    # 1. 初始化生成器
    # 注意：固定 random_seed 以确保结果可复现
    generator = MTHFVRPGenerator(
        num_loc=nums_customers, 
        variant_preset=variant, 
        vehicle_num=num_veh, 
        random_seed=42
    ) 

    # 2. 生成数据
    problem = generator(nums_problems)
    
    # 3. 生成问题签名 (Signature)    
    file_prefix = f"{variant}_node{nums_customers}_instance{nums_problems}"
    
    first_loc_start = problem['locs'][0][0].tolist()
    first_loc_end = problem['locs'][-1][0].tolist()
    
    sig = f"Signature [{file_prefix}]: {first_loc_start} -> {first_loc_end}"
    print(sig)

    # 4. 保存到文件
    save_path = os.path.join(output_dir, variant, f"{file_prefix}.jsonl")
    save_problems_to_jsonl(problem, save_path)

Generating Variants:   0%|          | 0/3 [00:00<?, ?it/s]

Initialized MTHFVRPGenerator with random_seed=42
Signature [cvrp_node50_instance1000]: [0.8822692632675171, 0.9150039553642273] -> [0.1464962363243103, 0.04784882068634033]
创建目录: ../data/cvrp


Generating Variants:  33%|███▎      | 1/3 [00:00<00:00,  3.31it/s]

成功保存 1000 个实例到: ../data/cvrp/cvrp_node50_instance1000.jsonl
Initialized MTHFVRPGenerator with random_seed=42
Signature [vrptw_node50_instance1000]: [0.8822692632675171, 0.9150039553642273] -> [0.1464962363243103, 0.04784882068634033]
创建目录: ../data/vrptw


Generating Variants:  67%|██████▋   | 2/3 [00:00<00:00,  2.92it/s]

成功保存 1000 个实例到: ../data/vrptw/vrptw_node50_instance1000.jsonl
Initialized MTHFVRPGenerator with random_seed=42
Signature [hfcvrp_node50_instance1000]: [0.8822692632675171, 0.9150039553642273] -> [0.1464962363243103, 0.04784882068634033]
创建目录: ../data/hfcvrp


Generating Variants: 100%|██████████| 3/3 [00:00<00:00,  3.07it/s]

成功保存 1000 个实例到: ../data/hfcvrp/hfcvrp_node50_instance1000.jsonl
